# 📚 Aula 2 — Estratégias de Chunking para Documentos Jurídicos
## MBA em RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Comparar na prática as principais estratégias de chunking usando um mesmo parágrafo de entrada.

**Duração estimada:** 90 minutos

---
### Diagrama do Fluxo desta Aula

```
Texto Jurídico de Entrada
         │
         ├──▶ 1. Fixed-Size Chunking
         ├──▶ 2. Recursive Character Chunking
         ├──▶ 3. Semantic Chunking
         └──▶ 4. Header-Based Chunking
                      │
                      ▼
         Tabela Comparativa de Resultados
```

> **ABNT:** LANGCHAIN. *Text Splitters*. Disponível em: <https://python.langchain.com/docs>. Acesso em: abr. 2026.

## 🔧 Célula 1 — Instalação das Dependências

Instalamos apenas o necessário para este notebook. O ambiente base já foi configurado na Aula 1.

In [ ]:
# ============================================================
# INSTALAÇÃO — Execute esta célula apenas uma vez por sessão
# Tempo estimado: 2–4 minutos no Google Colab
# ============================================================

!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q langchain-experimental  # Para SemanticChunker
!pip install -q sentence-transformers   # Modelo de embedding local
!pip install -q tiktoken               # Contador de tokens OpenAI-compatível

print("✅ Dependências instaladas com sucesso!")

## 📄 Célula 2 — Documento de Entrada (Texto Jurídico Real)

Usaremos um trecho baseado em decisão judicial real para testar todas as estratégias no **mesmo texto**. Isso permite comparação justa.

O texto representa um acórdão com:
- Ementa
- Relatório
- Fundamentação
- Dispositivo

In [ ]:
# ============================================================
# DOCUMENTO DE ENTRADA — Acórdão Jurídico Simulado
# Texto baseado em estrutura real de acórdãos brasileiros
# ============================================================

TEXTO_ACORDAO = """
EMENTA

HABEAS CORPUS. TRÁFICO DE DROGAS. PRISÃO PREVENTIVA. REQUISITOS LEGAIS. FUNDAMENTAÇÃO INIDÔNEA. EXCESSO DE PRAZO. CONSTRANGIMENTO ILEGAL CONFIGURADO. ORDEM CONCEDIDA.

1. A prisão preventiva constitui medida cautelar de natureza excepcional, somente admissível quando presentes os requisitos previstos nos artigos 312 e 313 do Código de Processo Penal, quais sejam: garantia da ordem pública, garantia da ordem econômica, conveniência da instrução criminal ou assegurar a aplicação da lei penal.

2. A fundamentação da prisão preventiva não pode se basear exclusivamente na gravidade abstrata do delito ou na mera reprodução dos termos legais, sendo necessária a indicação concreta de fatos que justifiquem a custódia cautelar.

3. O excesso de prazo na formação da culpa, quando não justificado por circunstâncias excepcionais ou pela complexidade da causa, configura constrangimento ilegal passível de ser sanado pela via do habeas corpus.

RELATÓRIO

Trata-se de habeas corpus com pedido de medida liminar impetrado por FULANO DE TAL, por intermédio de seu defensor constituído, em favor de SICRANO DE TAL, contra decisão proferida pelo Juízo da 3ª Vara Criminal da Comarca de São Paulo, que decretou a prisão preventiva do paciente nos autos do Processo nº 0001234-56.2024.8.26.0001, instaurado em razão de suposta prática do crime previsto no artigo 33 da Lei nº 11.343/2006 (Tráfico de Drogas).

O impetrante alega, em síntese, que: (a) a decisão que decretou a prisão preventiva carece de fundamentação idônea, limitando-se a reproduzir os termos da lei sem indicar elementos concretos que justifiquem a segregação cautelar; (b) o paciente está preso há mais de 120 (cento e vinte) dias sem que tenha sido concluída a instrução criminal, configurando excesso de prazo; (c) o paciente é primário, tem residência fixa e trabalho lícito comprovado, circunstâncias que afastam os fundamentos da prisão preventiva.

A autoridade coatora prestou informações confirmando a manutenção da decisão atacada, sustentando a necessidade da custódia cautelar em razão da gravidade do delito e da quantidade de droga apreendida.

O Ministério Público opinou pelo não conhecimento do writ e, subsidiariamente, pela denegação da ordem.

FUNDAMENTAÇÃO

A prisão preventiva é medida cautelar de ultima ratio, devendo ser decretada somente quando as demais medidas cautelares previstas no artigo 319 do Código de Processo Penal se mostrarem inadequadas ou insuficientes. A Constituição Federal de 1988, em seu artigo 5º, inciso LXI, estabelece que ninguém será preso senão em flagrante delito ou por ordem escrita e fundamentada de autoridade judiciária competente.

No caso em tela, verifica-se que a decisão que decretou a prisão preventiva do paciente limitou-se a afirmar que a medida era necessária para a garantia da ordem pública e para a conveniência da instrução criminal, sem, contudo, indicar elementos concretos do caso que justificassem tal conclusão. A mera referência à gravidade abstrata do crime de tráfico de drogas e à quantidade de entorpecentes apreendida não constitui fundamentação idônea para a decretação da custódia cautelar.

Nesse sentido, é pacífica a jurisprudência do Superior Tribunal de Justiça no sentido de que a gravidade em abstrato do delito não constitui fundamento idôneo para a manutenção da prisão preventiva. Precedentes: HC 123.456/SP, HC 654.321/RJ.

Ademais, constata-se que o paciente está preso há mais de 120 dias, sem que tenha havido justificativa razoável para a demora na conclusão da instrução criminal. O processo encontra-se pendente de oitiva de apenas 2 (duas) testemunhas arroladas pela defesa, circunstância que não justifica o excesso de prazo verificado.

DISPOSITIVO

Diante do exposto, CONCEDO a ordem de habeas corpus para determinar a imediata colocação do paciente SICRANO DE TAL em liberdade, salvo se por outro motivo deva permanecer preso, revogando-se a prisão preventiva decretada nos autos do Processo nº 0001234-56.2024.8.26.0001.

Comunique-se com urgência ao Juízo de origem e à autoridade coatora.

São Paulo, 15 de abril de 2025.
"""

# Contagem básica de métricas do texto
palavras = len(TEXTO_ACORDAO.split())
chars = len(TEXTO_ACORDAO)
paragrafos = len([p for p in TEXTO_ACORDAO.split('\n\n') if p.strip()])

print(f"📊 Métricas do texto de entrada:")
print(f"   Palavras:    {palavras}")
print(f"   Caracteres:  {chars}")
print(f"   Parágrafos:  {paragrafos}")

---
## 📦 Estratégia 1 — Fixed-Size Chunking

### Teoria

O `CharacterTextSplitter` divide o texto em chunks de tamanho fixo baseado em número de **caracteres**.

```
Texto: [A B C D E F G H I J K L M N O P]

chunk_size=6, overlap=2:
Chunk 1: [A B C D E F]
Chunk 2:         [E F G H I J]
Chunk 3:                 [I J K L M N]
                 ←2→     ←2→  (overlap)
```

**Vantagem:** Muito rápido, sem dependência de modelo de embedding.

**Desvantagem:** Pode cortar no meio de uma sentença ou parágrafo.

> 💡 **Dica jurídica:** Para artigos de lei numerados, use `separator="\n"` para garantir que cada artigo fique em um chunk separado.

In [ ]:
# ============================================================
# ESTRATÉGIA 1: Fixed-Size Chunking com CharacterTextSplitter
# ============================================================
from langchain.text_splitter import CharacterTextSplitter

# Configuração do splitter
# chunk_size=800: ~500 palavras — bom para parágrafos jurídicos médios
# chunk_overlap=150: 19% de overlap — garante continuidade entre chunks
# separator="\n\n": prefere quebrar em parágrafos (dupla newline)
fixed_splitter = CharacterTextSplitter(
    separator="\n\n",      # Tenta quebrar em parágrafos primeiro
    chunk_size=800,         # Máximo de 800 caracteres por chunk
    chunk_overlap=150,      # 150 caracteres de sobreposição
    length_function=len,    # Usa len() para medir (caracteres)
    is_separator_regex=False  # Separator é literal, não regex
)

# Aplicando o split no texto
chunks_fixed = fixed_splitter.split_text(TEXTO_ACORDAO)

# ---- Exibindo resultados ----
print(f"📦 FIXED-SIZE CHUNKING")
print(f"{'='*60}")
print(f"Parâmetros: chunk_size=800, overlap=150, sep='\\n\\n'")
print(f"Total de chunks gerados: {len(chunks_fixed)}")
print()

for i, chunk in enumerate(chunks_fixed, 1):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    # Mostra os primeiros e últimos 100 caracteres para visualizar fronteiras
    if len(chunk) > 200:
        print(f"  INÍCIO: {chunk[:100].strip()!r}")
        print(f"  FIM:    {chunk[-100:].strip()!r}")
    else:
        print(f"  CONTEÚDO: {chunk.strip()!r}")
    print()

### Tabela de Parâmetros — Fixed-Size

| `chunk_size` | `chunk_overlap` | Chunks Gerados | Melhor para |
|---|---|---|---|
| 200 | 20 | Muitos (15–20) | Artigos curtos, incisos |
| 500 | 50 | Médio (8–12) | Parágrafos simples |
| **800** | **150** | **Padrão (5–8)** | **Acórdãos, peças processuais** |
| 1200 | 200 | Poucos (3–5) | Relatórios longos |
| 2000 | 400 | Muito poucos (1–3) | Documentos muito longos |

In [ ]:
# ============================================================
# EXPERIMENTO: Impacto do chunk_size no número de chunks
# ============================================================
import pandas as pd

resultados = []
configs = [
    (200, 20),
    (500, 50),
    (800, 150),
    (1200, 200),
    (2000, 400),
]

for size, overlap in configs:
    splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=size,
        chunk_overlap=overlap,
        length_function=len
    )
    chunks = splitter.split_text(TEXTO_ACORDAO)
    avg_size = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    resultados.append({
        "chunk_size": size,
        "chunk_overlap": overlap,
        "n_chunks": len(chunks),
        "avg_chars": int(avg_size),
        "min_chars": min(len(c) for c in chunks),
        "max_chars": max(len(c) for c in chunks)
    })

df = pd.DataFrame(resultados)
print("\n📊 Impacto dos parâmetros no chunking:")
print(df.to_string(index=False))

---
## 🔁 Estratégia 2 — Recursive Character Chunking

### Teoria

O `RecursiveCharacterTextSplitter` tenta dividir usando uma **hierarquia de separadores**. Se o texto ainda for grande após usar o primeiro separador, tenta o próximo.

```
HIERARQUIA DE SEPARADORES (ordem de preferência):

1. "\n\n"  → Parágrafos     (máximo respeito à estrutura)
2. "\n"    → Linhas
3. " "     → Palavras
4. ""      → Caracteres     (mínimo respeito — último recurso)

Exemplo jurídico:

Texto: "Art. 5º ...parágrafo longo...\n\nArt. 6º ...outro parágrafo..."
→ Separa no \n\n → dois chunks respeitando artigos
```

> 💡 **Por que é melhor para textos jurídicos?** Porque preserva a coesão dos artigos, parágrafos e incisos. Um artigo do Código Penal raramente será cortado no meio.

In [ ]:
# ============================================================
# ESTRATÉGIA 2: Recursive Character Text Splitter
# ============================================================
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Separadores hierárquicos customizados para texto jurídico
# Adicionamos separadores típicos de documentos jurídicos brasileiros
SEPARADORES_JURIDICOS = [
    "\n\n",        # Parágrafos (nível mais alto)
    "\n",          # Linhas
    ". ",          # Final de sentença
    "; ",          # Ponto e vírgula (comum em incisos)
    ", ",          # Vírgula
    " ",           # Palavras
    "",            # Caracteres (último recurso)
]

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=SEPARADORES_JURIDICOS,
    chunk_size=800,          # Mesmo tamanho do Fixed para comparação justa
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=False
)

chunks_recursive = recursive_splitter.split_text(TEXTO_ACORDAO)

print(f"🔁 RECURSIVE CHARACTER CHUNKING")
print(f"{'='*60}")
print(f"Separadores: {SEPARADORES_JURIDICOS[:4]} + ...")
print(f"Total de chunks gerados: {len(chunks_recursive)}")
print()

for i, chunk in enumerate(chunks_recursive, 1):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    if len(chunk) > 200:
        print(f"  INÍCIO: {chunk[:100].strip()!r}")
        print(f"  FIM:    {chunk[-100:].strip()!r}")
    else:
        print(f"  CONTEÚDO: {chunk.strip()!r}")
    print()

In [ ]:
# ============================================================
# COMPARAÇÃO VISUAL: Fixed vs Recursive
# Verificar onde cada estratégia "quebrou" o texto
# ============================================================

print("🔍 COMPARAÇÃO: Fixed vs Recursive")
print("="*60)
print(f"{'Métrica':<30} {'Fixed':>10} {'Recursive':>10}")
print("-"*50)
print(f"{'Total de chunks':<30} {len(chunks_fixed):>10} {len(chunks_recursive):>10}")

avg_fixed = sum(len(c) for c in chunks_fixed) / len(chunks_fixed)
avg_rec = sum(len(c) for c in chunks_recursive) / len(chunks_recursive)
print(f"{'Tamanho médio (chars)':<30} {avg_fixed:>10.0f} {avg_rec:>10.0f}")

# Verifica quantos chunks terminam no meio de uma sentença
def conta_cortes_ruins(chunks):
    """Conta chunks que terminam sem ponto final (potencial corte ruim)"""
    return sum(1 for c in chunks if c.strip() and not c.strip().endswith(('.', '!', '?', ':')))

cortes_fixed = conta_cortes_ruins(chunks_fixed)
cortes_rec = conta_cortes_ruins(chunks_recursive)
print(f"{'Chunks com corte ruim':<30} {cortes_fixed:>10} {cortes_rec:>10}")
print()
print("💡 Menos cortes ruins = melhor preservação de contexto")

---
## 🧠 Estratégia 3 — Semantic Chunking

### Teoria

O `SemanticChunker` usa um modelo de embedding para detectar **mudanças semânticas** no texto e só quebra o chunk quando o assunto muda significativamente.

```
PROCESSO:
1. Divide texto em sentenças individuais
2. Gera embedding para cada sentença
3. Calcula distância coseno entre sentenças adjacentes
4. Identifica "breakpoints" onde a distância supera o threshold
5. Cria chunks agrupando sentenças do mesmo tema

Visualização:
S1──S2──S3  ||  S4──S5  ||  S6──S7──S8──S9
  Prisão       Habeas      Fundamentação
  Preventiva   Corpus      do Juiz
```

> ⚠️ **Custo:** O Semantic Chunking é ~10–50x mais lento que os métodos anteriores porque precisa calcular embeddings para cada sentença.

In [ ]:
# ============================================================
# ESTRATÉGIA 3: Semantic Chunking
# Usa embeddings locais (HuggingFace) para evitar custo de API
# ============================================================
from langchain_experimental.text_splitter import SemanticChunker
import os, requests
from dotenv import load_dotenv
load_dotenv()

OLLAMA_BASE_URL   = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')

# Estratégia: usar o BGE-M3 servido pelo Ollama da Aula 1 (padrão da Aula 2).
# Fallback: HuggingFaceEmbeddings(BAAI/bge-m3) local se o Ollama estiver fora.
print('⏳ Inicializando embeddings para Semantic Chunking...')

embeddings = None
try:
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3).raise_for_status()
    from langchain_ollama import OllamaEmbeddings
    embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    _ = embeddings.embed_query('teste')
    print(f'✅ OllamaEmbeddings OK | modelo={OLLAMA_EMBED_MODEL} @ {OLLAMA_BASE_URL}')
except Exception as e:
    print(f'⚠️  Ollama indisponível ({e}). Caindo para HuggingFace BAAI/bge-m3.')
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name='BAAI/bge-m3',
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True},
    )
    print('✅ HuggingFaceEmbeddings(BAAI/bge-m3) carregado')

# breakpoint_threshold_type="percentile":
#   Quebra o chunk quando a distância semântica está no percentil 85
#   (ou seja: quebra apenas nos 15% maiores saltos semânticos)
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",  # Opções: percentile, standard_deviation, interquartile
    breakpoint_threshold_amount=85,          # Percentil — ajuste para mais/menos chunks
    buffer_size=1                            # Sentenças de contexto em cada lado do breakpoint
)

print("⏳ Aplicando semantic chunking...")
chunks_semantic = semantic_chunker.split_text(TEXTO_ACORDAO)
print(f"✅ Semantic chunking concluído!")
print()
print(f"🧠 SEMANTIC CHUNKING")
print(f"{'='*60}")
print(f"Total de chunks gerados: {len(chunks_semantic)}")
print()

for i, chunk in enumerate(chunks_semantic, 1):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(f"  Início: {chunk[:120].strip()!r}")
    if len(chunk) > 200:
        print(f"  Fim:    {chunk[-80:].strip()!r}")
    print()

In [ ]:
# ============================================================
# EXPERIMENTO: Impacto do threshold no Semantic Chunking
# ============================================================
print("🔬 Experimento: Variando o breakpoint_threshold_amount")
print("="*60)

thresholds = [70, 80, 85, 90, 95]
for thresh in thresholds:
    sc = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=thresh,
    )
    chunks = sc.split_text(TEXTO_ACORDAO)
    avg = sum(len(c) for c in chunks)/len(chunks) if chunks else 0
    print(f"  Threshold={thresh}%: {len(chunks):2d} chunks | avg={avg:.0f} chars")

print()
print("💡 Threshold mais alto = menos quebras = chunks maiores")
print("   Para textos jurídicos densos: 85–90 costuma funcionar bem")

---
## 🏗️ Estratégia 4 — Header-Based Chunking (MarkdownHeaderTextSplitter)

### Teoria

Usa os **headers** do documento (H1, H2, H3) para criar chunks que carregam metadados de hierarquia:

```
# EMENTA          → metadado: {"Header 1": "EMENTA"}
texto da ementa   → chunk com metadado da seção

# RELATÓRIO       → metadado: {"Header 1": "RELATÓRIO"}
texto do relat.   → chunk com metadado da seção

## Das Alegações  → metadado: {"Header 1": "RELATÓRIO", "Header 2": "Das Alegações"}
```

**Vantagem:** O metadado de seção enriquece o retrieval — você pode filtrar por `Header 1 == "FUNDAMENTAÇÃO"`.

In [ ]:
# ============================================================
# ESTRATÉGIA 4: Header-Based Chunking
# Requer que o documento tenha marcação de headers
# ============================================================
from langchain.text_splitter import MarkdownHeaderTextSplitter

# Convertemos o texto para formato Markdown com headers
# (em produção, o Docling faz isso automaticamente)
TEXTO_MARKDOWN = """
# EMENTA

HABEAS CORPUS. TRÁFICO DE DROGAS. PRISÃO PREVENTIVA. REQUISITOS LEGAIS. FUNDAMENTAÇÃO INIDÔNEA. EXCESSO DE PRAZO. CONSTRANGIMENTO ILEGAL CONFIGURADO. ORDEM CONCEDIDA.

1. A prisão preventiva constitui medida cautelar de natureza excepcional, somente admissível quando presentes os requisitos previstos nos artigos 312 e 313 do Código de Processo Penal.

2. A fundamentação da prisão preventiva não pode se basear exclusivamente na gravidade abstrata do delito ou na mera reprodução dos termos legais.

3. O excesso de prazo na formação da culpa configura constrangimento ilegal passível de ser sanado pela via do habeas corpus.

# RELATÓRIO

## Identificação do Processo

Trata-se de habeas corpus impetrado em favor de SICRANO DE TAL, contra decisão que decretou prisão preventiva no Processo nº 0001234-56.2024.8.26.0001, por suposta prática do artigo 33 da Lei nº 11.343/2006 (Tráfico de Drogas).

## Das Alegações do Impetrante

O impetrante alega: (a) decisão sem fundamentação idônea; (b) excesso de prazo de 120 dias; (c) paciente primário com residência e trabalho fixos.

## Da Manifestação do Ministério Público

O Ministério Público opinou pelo não conhecimento do writ e, subsidiariamente, pela denegação da ordem.

# FUNDAMENTAÇÃO

## Da Prisão Preventiva como Medida Excepcional

A prisão preventiva é medida cautelar de ultima ratio, devendo ser decretada somente quando as demais medidas cautelares previstas no artigo 319 do CPP se mostrarem inadequadas ou insuficientes.

## Da Fundamentação Inidônea

No caso em tela, a decisão limitou-se a afirmar a necessidade para garantia da ordem pública, sem indicar elementos concretos. A jurisprudência do STJ é pacífica: HC 123.456/SP, HC 654.321/RJ.

## Do Excesso de Prazo

O paciente está preso há mais de 120 dias sem justificativa para a demora. Processo pendente de oitiva de apenas 2 testemunhas.

# DISPOSITIVO

CONCEDO a ordem de habeas corpus para determinar a imediata colocação do paciente em liberdade, revogando-se a prisão preventiva.
"""

# Define os headers que serão usados como separadores
headers_to_split_on = [
    ("#", "Seção"),      # H1 → campo "Seção" no metadado
    ("##", "Subseção"),  # H2 → campo "Subseção" no metadado
    ("###", "Item"),     # H3 → campo "Item" no metadado
]

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False  # Mantém o header no conteúdo do chunk
)

chunks_header = header_splitter.split_text(TEXTO_MARKDOWN)

print(f"🏗️ HEADER-BASED CHUNKING")
print(f"{'='*60}")
print(f"Total de chunks gerados: {len(chunks_header)}")
print()

for i, doc in enumerate(chunks_header, 1):
    print(f"--- Chunk {i} ---")
    print(f"  Metadados: {doc.metadata}")
    print(f"  Conteúdo ({len(doc.page_content)} chars):")
    print(f"  {doc.page_content[:150].strip()!r}")
    print()

In [ ]:
# ============================================================
# DEMONSTRAÇÃO: Filtragem por metadado de seção
# Mostra o poder do header-based chunking para retrieval preciso
# ============================================================
print("🎯 FILTRAGEM POR SEÇÃO — Poder do Header Chunking")
print("="*60)
print()

# Simula uma query específica: "Quero apenas chunks da FUNDAMENTAÇÃO"
secao_alvo = "FUNDAMENTAÇÃO"
chunks_fundamentacao = [
    doc for doc in chunks_header
    if doc.metadata.get("Seção", "") == secao_alvo
]

print(f"Chunks da seção '{secao_alvo}': {len(chunks_fundamentacao)}")
for i, doc in enumerate(chunks_fundamentacao, 1):
    print(f"\n  Chunk {i} | Subseção: {doc.metadata.get('Subseção', 'N/A')}")
    print(f"  {doc.page_content[:200].strip()!r}")

print()
print("💡 Em produção: use filtros de metadado no vector store")
print("   para buscar APENAS em seções relevantes (ex: só FUNDAMENTAÇÃO)")

---
## 📊 Comparativo Final das 4 Estratégias

In [ ]:
# ============================================================
# TABELA COMPARATIVA FINAL — Todas as Estratégias
# ============================================================
import pandas as pd

def stats_chunks(chunks, tipo):
    """Calcula estatísticas de um conjunto de chunks."""
    # Normaliza para lista de strings
    textos = [c.page_content if hasattr(c, 'page_content') else c for c in chunks]
    sizes = [len(t) for t in textos]
    return {
        "Estratégia": tipo,
        "N° Chunks": len(textos),
        "Média (chars)": f"{sum(sizes)/len(sizes):.0f}",
        "Mín (chars)": min(sizes),
        "Máx (chars)": max(sizes),
        "Preserva\nEstrutura": "✓" if tipo == "Header-Based" else "✗",
        "Velocidade": {"Fixed": "⚡⚡⚡", "Recursive": "⚡⚡", "Semantic": "⚡", "Header-Based": "⚡⚡"}.get(tipo, "-"),
        "Requer\nEmbedding": "Sim" if tipo == "Semantic" else "Não",
    }

dados = [
    stats_chunks(chunks_fixed, "Fixed"),
    stats_chunks(chunks_recursive, "Recursive"),
    stats_chunks(chunks_semantic, "Semantic"),
    stats_chunks(chunks_header, "Header-Based"),
]

df = pd.DataFrame(dados)
print("📊 COMPARATIVO FINAL — Mesmo texto, 4 estratégias")
print("="*80)
print(df.to_string(index=False))
print()
print("📌 RECOMENDAÇÃO para contexto jurídico:")
print("   • Textos uniformes (artigos de lei): Fixed ou Recursive")
print("   • Acórdãos estruturados: Header-Based (após Docling)")
print("   • Laudos e narrativas: Semantic")
print("   • Produção geral: Recursive (melhor equilíbrio)")

---
## 🏷️ Bônus — Adicionando Metadados a Documents

Em produção, **sempre** adicione metadados aos chunks. Isso permite:
- Citar a fonte na resposta gerada
- Filtrar por tribunal, data, tipo de documento
- Auditar de onde veio cada informação

In [ ]:
# ============================================================
# BOAS PRÁTICAS: Chunks com Metadados Ricos
# ============================================================
from langchain.schema import Document

# Metadados do documento original
metadata_base = {
    "fonte": "TJ-SP",
    "tipo_documento": "acórdão",
    "numero_processo": "0001234-56.2024.8.26.0001",
    "tribunal": "TJSP",
    "data_julgamento": "2025-04-15",
    "assunto": "habeas corpus - tráfico de drogas - prisão preventiva",
    "relator": "Des. Fulano de Tal",
    "camara": "7ª Câmara Criminal"
}

# Cria Documents com metadados enriquecidos
documents_com_meta = []
for i, chunk_text in enumerate(chunks_recursive):
    # Adiciona metadado específico de cada chunk
    meta = metadata_base.copy()
    meta["chunk_id"] = i
    meta["chunk_total"] = len(chunks_recursive)
    meta["chars"] = len(chunk_text)

    # Detecta seção automaticamente (heurística simples)
    for secao in ["EMENTA", "RELATÓRIO", "FUNDAMENTAÇÃO", "DISPOSITIVO"]:
        if secao in chunk_text.upper():
            meta["secao"] = secao
            break
    else:
        meta["secao"] = "DESCONHECIDA"

    doc = Document(
        page_content=chunk_text,
        metadata=meta
    )
    documents_com_meta.append(doc)

# Exibe um exemplo de Document com metadados
print("📋 Exemplo de Document com Metadados Enriquecidos:")
print("="*60)
doc_exemplo = documents_com_meta[0]
print(f"Metadados: {doc_exemplo.metadata}")
print(f"Conteúdo ({len(doc_exemplo.page_content)} chars):")
print(f"{doc_exemplo.page_content[:200]}...")
print()
print(f"Total de Documents criados: {len(documents_com_meta)}")

---
## ✅ Resumo da Aula

| Estratégia | Use quando |
|---|---|
| **Fixed** | Textos uniformes, processamento em lote rápido |
| **Recursive** | Maioria dos textos jurídicos (melhor default) |
| **Semantic** | Laudos, relatórios narrativos, qualidade máxima |
| **Header-Based** | PDFs estruturados após conversão com Docling |

**Próximos passos:** Notebook `EXEMPLO2_Docling_Ingestao.ipynb`